# Fórmulas e Contas — Análise Bracadas/Pernadas (Parkinson vs Controlo)

Este notebook documenta, com fórmulas e código executável, todas as contas feitas até agora no
pipeline (extração de features por ciclo, índice de assimetria, agrupamento PD/Controlo,
variabilidade e testes de hipótese), e as novas contas a fazer a seguir (variabilidade de força,
SPARC e velocidade).

Referências de código-fonte: `data_analysis_medium.py`, `analysis_results.py`,
`analysis_asymmetry.py`, `build_isolated_summary.py`.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
pd.set_option('display.width', 200)

## 1. Segmentação de ciclos (braçada/pernada)

Cada ciclo é delimitado por picos detetados num único eixo fixo por membro
(Y para braços, X para pernas — método Ödek & Özcan, 2023), após filtro passa-baixo
Butterworth de 3ª ordem a 2 Hz. Os limites do ciclo são os pontos médios entre picos
consecutivos. O primeiro e o último ciclo de cada exercício são excluídos da extração de
features (fase de aceleração/desaceleração incompleta).

Se o eixo fixo não encontrar pelo menos 3 picos (`FIXED_AXIS_MIN_PEAKS = 3`), o código
recorre a um método antigo de fallback (multi-eixo, passa-banda 0.3–2/3 Hz, escolhendo o
sinal com mais picos entre magnitude/x/y/z e as suas versões em valor absoluto).

## 2. Métricas por ciclo (ficheiro `cycles_medium.csv`)

Para cada ciclo, a partir do segmento de aceleração 3D `accel[inicio:fim]` (unidades **g**):

**Magnitude e aceleração dinâmica** (remove a componente quase-constante de gravidade/orientação):

$$\text{magnitude}_i = \sqrt{x_i^2 + y_i^2 + z_i^2}$$
$$\text{dynamic}_i = \text{magnitude}_i - \overline{\text{magnitude}}$$

**Duração**: `duration_s = n_amostras / fs`

**Mechanical Load** (proxy de força/vigor — soma da aceleração dinâmica absoluta):
$$\text{Mechanical\_Load} = \sum_i |\text{dynamic}_i|$$

**Peak Acceleration**: $\max_i |\text{dynamic}_i|$

**RMS Acceleration**: $\sqrt{\frac{1}{n}\sum_i \text{dynamic}_i^2}$

**SPARC (suavidade)** — Spectral Arc Length, calculado sobre a densidade espectral de
potência (Welch) do sinal `dynamic`:
$$\text{amp\_norm}(f) = \frac{\sqrt{PSD(f)}}{\max\sqrt{PSD(f)}}, \qquad
\text{SPARC} = -\sum \sqrt{(\Delta f)^2 + (\Delta \, \text{amp\_norm})^2}$$
(mais próximo de 0 = mais suave; mais negativo = menos suave)

**Velocidade** (estimativa por integração; NÃO é deslocamento real, é relativa):
1. Converter de g para m/s²: `dynamic_ms2 = dynamic * 9.80665`
2. Integração trapezoidal: `velocity_raw = cumulative_trapezoid(dynamic_ms2, dx=1/fs)`
3. Correção de deriva (detrend linear): `velocity = detrend(velocity_raw)`
4. `Avg_Velocity = mean(|velocity|)`, `Peak_Velocity = max(|velocity|)`

In [ ]:
# Reprodução em código do bloco de cálculo (ver data_analysis_medium.py, linhas ~340-393)
from scipy.signal import welch, detrend
from scipy.integrate import cumulative_trapezoid

GRAVITY_MS2 = 9.80665

def metricas_por_ciclo(accel_xyz, fs):
    """accel_xyz: array (n_amostras, 3) em unidades g. Retorna dict de métricas do ciclo."""
    magnitude = np.linalg.norm(accel_xyz, axis=1)
    dynamic = magnitude - np.mean(magnitude)

    mechanical_load = float(np.sum(np.abs(dynamic)))
    peak_accel = float(np.max(np.abs(dynamic)))
    rms_accel = float(np.sqrt(np.mean(dynamic**2)))

    f, psd = welch(dynamic, fs=fs, nperseg=min(len(dynamic), 1024))
    amp_norm = np.sqrt(psd) / (np.max(np.sqrt(psd)) + 1e-8)
    sparc = float(-np.sum(np.sqrt(np.diff(f)**2 + np.diff(amp_norm)**2)))

    if len(dynamic) >= 2:
        dynamic_ms2 = dynamic * GRAVITY_MS2
        velocity_raw = cumulative_trapezoid(dynamic_ms2, dx=1.0/fs, initial=0.0)
        velocity = detrend(velocity_raw)
        avg_velocity = float(np.mean(np.abs(velocity)))
        peak_velocity = float(np.max(np.abs(velocity)))
    else:
        avg_velocity = peak_velocity = None

    return dict(mechanical_load=mechanical_load, peak_accel=peak_accel, rms_accel=rms_accel,
                sparc=sparc, avg_velocity=avg_velocity, peak_velocity=peak_velocity)

## 3. Índice de Assimetria (ficheiro `asymmetry_results.csv`)

Para qualquer métrica $X$, comparando lado direito (R) e esquerdo (L):

$$AI(\%) = \frac{X_R - X_L}{X_R + X_L} \times 100$$

$AI=0$ → simetria perfeita; $AI>0$ → lado direito domina; $AI<0$ → lado esquerdo domina.
Também calculado separadamente para os terços inicial/médio/final do exercício
(`ai_early`, `ai_mid`, `ai_late`) para detetar tendência de fadiga assimétrica.

In [ ]:
def indice_assimetria(x_right, x_left):
    total = x_right + x_left
    if total == 0:
        return None
    return (x_right - x_left) / total * 100.0

## 4. Definição dos grupos PD vs Controlo

Os grupos (PD / Controlo) e as exclusões estão em `data/participants.json` (README, secção 3),
escrito à mão a partir do registo de participantes do estudo; a célula seguinte vai buscá-los a
`participants.py`. Só contam os participantes com dados de movimento extraídos
(`cycles_by_limb.csv`). A Control 2 está marcada como excluída — os
ficheiros `.cwa` dela são bit-a-bit idênticos aos da Control 1, o que constituiria
pseudo-replicação num teste-t.

In [ ]:
# Grupos e exclusoes (Control 2: mesmos ficheiros .cwa, byte a byte, que a Control 1)
# em data/participants.json
from participants import PD_ALL, CONTROL_ALL, GROUP_MAP

print(f'PD: n={len(PD_ALL)}   Control: n={len(CONTROL_ALL)}')

## 5. Métrica de variabilidade (CV%) por paciente × exercício

Para cada paciente e cada exercício (`exercise_label`), agregando todos os ciclos desse
membro (D+E combinados, via `cycles_by_limb.csv`):

$$CV\%(X) = \frac{\sigma(X)}{\mu(X)} \times 100$$

Onde $X$ é a métrica escolhida (duração, força, SPARC ou velocidade) ao longo de todos os
ciclos desse paciente nesse exercício. Usa-se CV% (não o desvio-padrão bruto) porque
normaliza pela magnitude média de cada paciente, tornando a comparação entre pessoas com
braçadas/pernadas de tamanhos diferentes justa.

In [ ]:
def cv_pct_por_paciente_exercicio(cycles_by_limb_df, coluna, grupo_map):
    df = cycles_by_limb_df[cycles_by_limb_df['patient'].isin(grupo_map.keys())].copy()
    df['group'] = df['patient'].map(grupo_map)
    agg = df.groupby(['patient','group','exercise_label'])[coluna].agg(['count','mean','std']).reset_index()
    agg['cv_pct'] = agg['std'] / agg['mean'] * 100
    return agg

## 6. Teste de hipótese: PD vs Controlo

**Hipótese nula** ($H_0$): não há diferença na variabilidade (CV%) de X entre o grupo PD e
o grupo de Controlo, para um dado exercício.

**Teste**: t-test de Welch (variâncias não assumidas iguais — mais robusto com grupos
pequenos e desiguais):

$$t = \frac{\bar{X}_{PD} - \bar{X}_{Control}}{\sqrt{\dfrac{s_{PD}^2}{n_{PD}} + \dfrac{s_{Control}^2}{n_{Control}}}}$$

**Tamanho de efeito** (Cohen's d, desvio-padrão combinado):

$$d = \frac{\bar{X}_{PD} - \bar{X}_{Control}}{\sqrt{\dfrac{s_{PD}^2 + s_{Control}^2}{2}}}$$

Interpretação de $d$: ~0.2 pequeno, ~0.5 médio, ~0.8 grande (Cohen, 1988).

In [ ]:
def teste_pd_vs_controlo(agg_df, exercicio):
    sub = agg_df[agg_df['exercise_label'] == exercicio]
    pd_vals = sub[sub['group']=='PD']['cv_pct'].dropna()
    ct_vals = sub[sub['group']=='Control']['cv_pct'].dropna()
    if len(pd_vals) < 2 or len(ct_vals) < 2:
        return dict(exercicio=exercicio, aviso='dados insuficientes', n_pd=len(pd_vals), n_control=len(ct_vals))
    t, p = stats.ttest_ind(pd_vals, ct_vals, equal_var=False)
    d = (pd_vals.mean() - ct_vals.mean()) / np.sqrt((pd_vals.std()**2 + ct_vals.std()**2) / 2)
    return dict(exercicio=exercicio, n_pd=len(pd_vals), media_pd=pd_vals.mean(), sd_pd=pd_vals.std(),
                n_control=len(ct_vals), media_control=ct_vals.mean(), sd_control=ct_vals.std(),
                t=t, p=p, cohen_d=d)

## 7. Resultado já obtido: variabilidade da DURAÇÃO do ciclo

Executa a análise completa (secções 4–6) para a métrica `duration_s`.

In [ ]:
cbl = pd.read_csv('cycles_by_limb.csv')

agg_duration = cv_pct_por_paciente_exercicio(cbl, 'duration_s', GROUP_MAP)

resultados_duracao = [teste_pd_vs_controlo(agg_duration, ex) for ex in ['Arm exercise','Leg exercise','Swim 1','Swim 2']]
pd.DataFrame(resultados_duracao)

## 8. NOVAS contas a fazer agora: variabilidade de Força, SPARC e Velocidade

Mesma lógica da secção 5–6, aplicada às colunas `mechanical_load`, `sparc` e
`avg_velocity_ms` de `cycles_by_limb.csv`, em vez de `duration_s`. Objetivo: verificar se
alguma destas métricas mostra maior separação entre os grupos PD e Controlo do que a
duração (que não mostrou diferença significativa nem effect size relevante).

In [ ]:
agg_forca = cv_pct_por_paciente_exercicio(cbl, 'mechanical_load', GROUP_MAP)
resultados_forca = [teste_pd_vs_controlo(agg_forca, ex) for ex in ['Arm exercise','Leg exercise','Swim 1','Swim 2']]
pd.DataFrame(resultados_forca)

In [ ]:
agg_sparc = cv_pct_por_paciente_exercicio(cbl, 'sparc', GROUP_MAP)
resultados_sparc = [teste_pd_vs_controlo(agg_sparc, ex) for ex in ['Arm exercise','Leg exercise','Swim 1','Swim 2']]
pd.DataFrame(resultados_sparc)

In [ ]:
agg_velocidade = cv_pct_por_paciente_exercicio(cbl, 'avg_velocity_ms', GROUP_MAP)
resultados_velocidade = [teste_pd_vs_controlo(agg_velocidade, ex) for ex in ['Arm exercise','Leg exercise','Swim 1','Swim 2']]
pd.DataFrame(resultados_velocidade)

## 9. Ressalvas importantes

- **N pequeno** (o n de cada grupo é impresso na secção 4, e o de cada exercício nas colunas
  `n_pd` e `n_control`) — poder estatístico modesto; um
  resultado não-significativo não prova ausência de diferença (erro Tipo II é plausível).
- **Split de dados para qualquer ML futuro deve ser por paciente**, nunca por ciclo/janela
  (evitar fuga de dados).
- **SPARC por ciclo individual tem pouca variância** (~-50 sempre) — CV% desta métrica pode
  ser dominado por ruído numérico mais do que por sinal clínico real.
- **Velocidade é uma estimativa relativa** (integração de acelerómetro, sem referência de
  posição absoluta) — comparações entre pacientes são válidas, valores absolutos não.
- Control 2 excluída do grupo de Controlo (duplicado da Control 1); alguns
  participantes registados não têm dados de movimento extraídos e ficam fora de qualquer
  análise.